# Audio Understanding

**Module:** 16 — Speech AI

Beyond transcripts: intent, emotion, events, diarization, and multimodal audio models.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- List audio understanding tasks beyond ASR
- Explain diarization and evaluation pitfalls
- Combine transcript + audio features for intent/risk
- Sketch multimodal audio→LLM request patterns


## Tasks Beyond Transcription

### Definition
Audio understanding extracts structure and semantics that plain text may miss — speakers, events, affect, intent.

### Why it matters
Meeting tools and safety systems need *who/when/what happened*, not only words.

### How it works
Task family: diarization, sound event detection, emotion/sentiment, language ID, speaker verify, multimodal audio LLMs.

### Intuition
Closed captions vs a director's commentary track.

### Pitfalls
- Using sentiment alone for HR decisions
- Ignoring overlapping speech

### When to use
Meetings, compliance, media indexing, call analytics.


### Task catalog

| Task | Output | Notes |
|------|--------|-------|
| Diarization | Who spoke when | Overlaps hard |
| Emotion / affect | Labels/scores | Culture-sensitive |
| Sound events | Alarms, applause… | Non-speech |
| Language ID | Locale | Route ASR |
| Speaker verify | Same/not same | Biometric rules |
| Audio LLM QA | Freeform answers | Needs grounding |

```mermaid
flowchart TB
  A[Audio] --> ASR[ASR]
  A --> DIA[Diarization]
  A --> EVT[Events]
  ASR --> M[Merge timeline]
  DIA --> M
  EVT --> M
  M --> LLM[Reasoning / analytics]
```


In [ ]:
# Demo 1: merge transcript words with speaker turns
words = [
    {"t0": 0.0, "t1": 0.4, "w": "hello"},
    {"t0": 0.5, "t1": 1.0, "w": "thanks"},
    {"t0": 1.2, "t1": 1.8, "w": "sure"},
]
turns = [{"spk": "A", "t0": 0.0, "t1": 1.1}, {"spk": "B", "t0": 1.1, "t1": 2.0}]

def label_words(words, turns):
    out = []
    for w in words:
        mid = 0.5 * (w["t0"] + w["t1"])
        spk = next((t["spk"] for t in turns if t["t0"] <= mid <= t["t1"]), "?")
        out.append({**w, "spk": spk})
    return out
print(label_words(words, turns))


In [ ]:
# Demo 2: diarization error rate ingredients (toy)
def overlap(a0,a1,b0,b1):
    return max(0.0, min(a1,b1) - max(a0,b0))

# confusion: hypothesized speaker mapped wrong on a segment
ref = [("A",0,2), ("B",2,4)]
hyp = [("X",0,2.2), ("Y",2.2,4)]
# after optimal mapping X->A, Y->B, compute missed overlap-ish error proxy
mapped = {"X":"A","Y":"B"}
err = 0.0; total = 4.0
for hs,h0,h1 in hyp:
    rs = mapped[hs]
    # time where hyp speaker timeline mismatches ref speaker
    for r_s,r0,r1 in ref:
        ov = overlap(h0,h1,r0,r1)
        if ov and r_s != rs:
            err += ov
print("approx confusion seconds", err, "rate", err/total)


## Diarization

### Definition
Diarization answers *who spoke when* — clustering speaker embeddings over time.

### Why it matters
Action items and analytics are wrong if speakers are swapped.

### How it works
VAD → embeddings → clustering/assignment → optional identification. Overlaps and short turns are hard.

### Intuition
Color-coding a transcript by voice.

### Pitfalls
- Assuming 2 speakers always
- Using diarization as legal-grade identification without verify

### When to use
Meetings, interviews, call centers.


In [ ]:
# Demo 3: risk cues from text+audio metadata
def call_risk(transcript: str, meta: dict) -> dict:
    score = 0
    reasons = []
    t = transcript.lower()
    if any(k in t for k in ("lawyer", "attorney", "sue")):
        score += 2; reasons.append("legal_language")
    if meta.get("anger_score", 0) > 0.7:
        score += 2; reasons.append("high_anger")
    if meta.get("overlap_ratio", 0) > 0.25:
        score += 1; reasons.append("crosstalk")
    return {"score": score, "route": "human" if score >= 3 else "bot", "reasons": reasons}
print(call_risk("I will call my lawyer", {"anger_score": 0.8, "overlap_ratio": 0.1}))
print(call_risk("thanks bye", {"anger_score": 0.1, "overlap_ratio": 0.0}))


## Multimodal Audio Models

### Definition
Audio-capable LLMs accept sound (or features) plus text instructions to summarize, answer, or extract.

### Why it matters
Useful when prosody/events matter, or for long audio QA without perfect ASR.

### How it works
Send audio URL/base64 + prompt; still validate factual claims against ASR when stakes are high.

### Intuition
A model that can listen to the podcast, not only the transcript.

### Pitfalls
- Skipping transcript fallback when audio LLM fails
- Feeding hours of audio without chunking

### When to use
Meeting QA, media analysis, voice agent sensing.


In [ ]:
# Demo 4: audio LLM request sketch
import json
req = {
    "model": "gpt-4o-audio-preview",
    "messages": [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Summarize action items and list speakers if clear."},
            {"type": "input_audio", "input_audio": {"data": "YOUR_BASE64_AUDIO", "format": "wav"}},
        ],
    }],
}
print(json.dumps(req, indent=2)[:400])
print("OPENAI_API_KEY=YOUR_OPENAI_API_KEY")


In [ ]:
# Demo 5: chunk long audio timeline for understanding
def chunk_timeline(duration_s, chunk=30, overlap=5):
    out, t = [], 0.0
    while t < duration_s:
        out.append((t, min(duration_s, t+chunk)))
        t += chunk - overlap
    return out
print(chunk_timeline(100))


In [ ]:
# Demo 6: emotion label smoothing
def smooth(labels, k=3):
    # majority over sliding window
    out = []
    for i in range(len(labels)):
        window = labels[max(0,i-k+1):i+1]
        out.append(max(set(window), key=window.count))
    return out
print(smooth(["neu","neu","ang","ang","ang","neu","neu"]))


### Pitfalls

| Pitfall | Why bad | Mitigation |
|---------|---------|------------|
| Affect → HR action | Bias/legal | Human review; careful policy |
| Overlap ignored | Lost words | Overlap-aware ASR/diarization |
| ID without consent | Biometric law | Verify vs diarize separation |
| No grounding | Audio LLM invents | Cite transcript spans |


### Checklist — Audio understanding

- [ ] Tasks explicitly scoped (not 'magic listen')
- [ ] Diarization evaluated on overlaps
- [ ] Policy for affect/biometric outputs
- [ ] Chunking strategy for long files
- [ ] Grounding to timestamps


### Try it yourself — Understanding

1. Extend Demo 1 to emit markdown dialogue script.
2. Add keyword event detector ('gunshot','glass') mock.
3. Define allowed uses for emotion scores in your org.

**Stretch:** Build a mini DER metric on synthetic turns.


### Try it yourself — Audio LLM

1. Design prompts that require timestamp citations.
2. Fallback: if audio model low-conf, use ASR summary.


## Knowledge Check

**Q1.** Diarization vs speaker verification?

<details><summary>Answer</summary>

Diarization clusters unknown speakers; verification checks a claimed identity against an enrollment.

</details>

**Q2.** Why chunk long audio?

<details><summary>Answer</summary>

Context/latency/cost limits; improves localization of answers.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `diarization` | Who spoke when |
| `DER` | Diarization error rate |
| `sound event` | Non-speech audio class |
| `affect` | Emotion/paralinguistic state |
| `audio LLM` | LLM accepting audio inputs |
| `overlap` | Simultaneous speech |


## Key Takeaways

- Understanding ≠ transcription alone
- Diarization enables usable meeting analytics
- Treat affect/biometrics as high-risk outputs
- Ground audio LLM answers to time spans


## Production Incident Patterns — audio understanding

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Users talk over bot | No barge-in / bad VAD | Tune endpointing; cancel TTS |
| High WER in field | Noise/codec mismatch | Denoise; match sample rate |
| Creepy voice clone | Weak consent policy | Watermark + allow-list |
| 800ms+ dead air | Cascaded STT→LLM→TTS | Speculative TTS; S2S; stream |
| Compliance scare | Raw audio retention | TTL + transcript-only default |

```
Voice control loop:
  mic -> VAD -> ASR partials -> NLU/LLM -> TTS stream -> speaker
                ^                | tools/HITL
                +-- transcripts/metrics/audit --+
```


In [ ]:
# Cross-cutting: never log raw secrets or full audio bytes
import hashlib, json

def audio_audit(user_id: str, wav_bytes: bytes, meta: dict) -> dict:
    return {
        "user_id": user_id,
        "sha256_16": hashlib.sha256(wav_bytes).hexdigest()[:16],
        "nbytes": len(wav_bytes),
        "meta": {k: v for k, v in meta.items() if k not in {"api_key", "authorization"}},
        "topic": "audio understanding",
    }

print(json.dumps(audio_audit("u1", b"RIFF....", {"model": "whisper", "api_key": "YOUR_OPENAI_API_KEY"})))


## Mini Case Study — audio understanding

**Scenario:** A support org replaces IVR menus with a voice agent. Pilot NPS soars.
**Month 2:** Accents under-served; callers interrupted mid-sentence; recordings retained 2 years.

**Retro questions**
1. What was the latency budget (ASR+LLM+TTS)?
2. Was barge-in tested with noisy headsets?
3. Retention: audio vs transcript vs redacted entities?
4. Which intents require human transfer?

**Design rule:** conversational voice is a real-time distributed system — optimize the path, not only model quality.


In [ ]:
# Cross-cutting: latency budget checker
from dataclasses import dataclass

@dataclass
class VoiceBudget:
    asr_ms: int = 300
    llm_first_token_ms: int = 400
    tts_first_audio_ms: int = 200
    network_ms: int = 100
    def total(self): return self.asr_ms + self.llm_first_token_ms + self.tts_first_audio_ms + self.network_ms
    def ok(self, sla=900): return self.total() <= sla

b = VoiceBudget()
print("audio understanding", "total_ms", b.total(), "ok", b.ok())
print("tight", VoiceBudget(500, 600, 300, 150).ok())


### Try it yourself — audio understanding ops

1. Draft an on-call runbook bullet list for audio understanding when p95 turn latency > SLA.
2. Sketch metrics: WER proxy, barge-in rate, transfer rate, audio retention age.
